# STMD: tuned two-pathway experiment

Run this notebook from the repository root. It loads the **original 1920×1080** footage, downsamples with `INTER_AREA`, generates a 1 m subpixel test target, and evaluates the **tuned** detector (no competing-hypothesis tracker). This is a synthetic demonstration, **not real event-camera data**.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import benchmark as b
from stmd_model import apparent_diameter_px

source, fps = b.load_input()
base = b.make_20s(source)
print(f"Source: {len(source)} frames at {fps} fps, detector frames: {base.shape}")
print("6 mm at 1 m:", round(apparent_diameter_px(6,1),3),"sensor pixels")

## Target injection and calibration
The dot is inserted **after** `INTER_AREA` downsampling using a **12× supersampled disk** and a small `σ=0.42 px` coverage-mask blur. The two detector thresholds are calibrated from the dot-free control (frames 25–99); no ground-truth coordinate enters inference.

In [ ]:
thresholds=b.calibrate(base)
with_dot,gt,d=b.apply_target(base,1.0)
print('Global pathway thresholds:',thresholds)
fig,axes=plt.subplots(1,2,figsize=(8,4))
axes[0].imshow(base[130][:,:,::-1]);axes[0].set_title('No target')
axes[1].imshow(with_dot[130][:,:,::-1]);axes[1].set_title('Subpixel synthetic target')
for a in axes:a.axis('off')
plt.tight_layout()

## Detection, false alarms, and localization accuracy
A correct frame requires an accepted global peak **within 8 pixels** of the injected ground truth. Evaluation uses frames 100–249, within the original forward sequence.

In [ ]:
df,_=b.evaluate_frames(with_dot,thresholds,gt=gt,make_vis=False)
control,_=b.evaluate_frames(base,thresholds,make_vis=False)
window=df.iloc[100:250]
print(f"1 m correct detections: {window.correct.mean():.1%}")
print(f"No-target false-alarm rate: {control.detected.iloc[100:250].mean():.1%}")
fig,ax=plt.subplots(figsize=(8,3))
ax.plot(df.time_s,df.confidence,lw=1,label='Uncalibrated evidence score')
ax.set(xlabel='Time (s)',ylabel='Evidence');ax.legend();plt.show()

## HD demo and architecture

![Tuned neural architecture](../media/architecture.png)

[HD 1 m demo](../media/stmd_tuned_1m_HD_short.mp4) · [Methods](../docs/METHODS.md)

For the full reproducibility benchmark run `python benchmark.py` from the repository root.